# Troiani Tokenizer - Vocabulary Discovery

**Goal:** Find the optimal BPE vocabulary (50,032 tokens) for a sub-1B LLM, concise and production-ready.

**Why 50,032?** 50,000 regular tokens + 32 special tokens (`<pad>`, `<bos>`, `<eos>`, `<unk>`, `<mask>`, plus 27 reserved).

We target **English / Spanish / Code (Python)**.


---
## 1. Data Sources

| Source | Domain | Access |
|---|---|---|
| FineWeb-Edu | English | `HuggingFaceFW/fineweb-edu` |
| Spanish Wikipedia / OSCAR | Spanish | `wikipedia` (20220301.es), `oscar-corpus/OSCAR-2301` |
| The Stack (Python) | Code | `bigcode/the-stack-dedup` (filter lang=Python) |
| The Pile | Mixed buffer | EleutherAI |


In [ ]:
# === Download & sample ===
from datasets import load_dataset

# ds = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)
# # save 100K docs to data/english.txt
pass


---
## 2. Experiments

### Exp 1: Vocab Size Sweep

Train on fixed mix (40% en, 30% es, 30% py). Test at 8K / 16K / 32K / **50K** / 64K.

Measure compression ratio (bytes/token) on held-out en, es, py.

**Decision:** 50K must compress >=15% better than 32K to justify extra 20M embedding params.


In [ ]:
# === EXP 1: Vocab size sweep ===
from troiani.tokenizer import TroianiTokenizer
import os

def compression_ratio(tok_path, test_path):
    t = TroianiTokenizer.from_pretrained(tok_path)
    with open(test_path) as f:
        text = f.read()
    return len(text.encode("utf-8")) / len(t.encode(text))

# for vs in [8000, 16000, 32000, 50032, 64000]:
#     train, evaluate, print table
pass


### Exp 2: Data Mix Sensitivity

Fix vocab=50K. Vary mix: A (50/25/25), B (33/33/33), C (60/20/20), D (40/40/20).

**Goal:** Minimize worst-case compression across domains.


### Exp 3: Merge Quality

Check common words/keywords are single tokens. Look for garbage merges.


In [ ]:
# === EXP 3: Merge quality ===
t = TroianiTokenizer.from_pretrained("resources/tokenizer/tokenizer_50032.json")
for w in ["the","que","def","return","import","class","None","\n","    ","self."]:
    ids = t.encode(w)
    print(f"{w:>10s} -> {len(ids)} tokens: {[t.decode([i]) for i in ids]}")


---
## 3. Decision

| Exp | Finding | Impact |
|---|---|---|
| 1. Size | 50K compresses ___% better than 32K | Worth it? |
| 2. Mix | Mix ___ is best | Use for final |
| 3. Merges | Any garbage? | Tune min_frequency |

**Final:** `resources/tokenizer/tokenizer_50032.json`


In [ ]:
# === FINAL: Train & verify ===
t = TroianiTokenizer(vocab_size=50032)
# t.train(files=[...], output_dir="resources/tokenizer")
# t2 = TroianiTokenizer.from_pretrained("resources/tokenizer/tokenizer_50032.json")
# print(f"Vocab: {len(t2)}")
pass
